In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
import os

API_KEY = "eb64fc98f390093d2056ab1314c1ba89deb63537745bfbfc14eca45af029966d" 

# Location ID for US Consulate HCMC 
LOCATION_ID = 3276359             

# Date range 
START_DATE = "2024-01-01"
END_DATE = datetime.now().strftime("%Y-%m-%d") 

# Folder to store monthly files
OUTPUT_FOLDER = "hcmc_air_data"

# Create folder if it doesn't exist
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)
    print(f"Created folder: {OUTPUT_FOLDER}")
else:
    print(f"Folder '{OUTPUT_FOLDER}' ready.")

Created folder: hcmc_air_data


In [6]:
# --- CELL 2: HELPER FUNCTIONS (UPDATED) ---

def get_location_sensors(location_id):
    """
    Step 1: Ask the Location which sensors it has.
    Returns a dictionary: {'pm25': 11357424, 'temperature': 11357401, ...}
    """
    url = f"https://api.openaq.org/v3/locations/{location_id}"
    headers = {"X-API-Key": API_KEY}
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Extract sensor list
        sensors = data['results'][0]['sensors']
        
        # Create map: name -> id
        sensor_map = {s['parameter']['name']: s['id'] for s in sensors}
        return sensor_map
        
    except Exception as e:
        print(f"❌ Error finding sensors: {e}")
        return {}

def fetch_sensor_measurements(sensor_id, start_date, end_date):
    """
    Step 2: Download data for ONE specific sensor ID.
    """
    url = f"https://api.openaq.org/v3/sensors/{sensor_id}/measurements"
    headers = {"X-API-Key": API_KEY}
    
    all_results = []
    page = 1
    
    while True:
        params = {
            "datetime_from": start_date, 
            "datetime_to": end_date,
            "limit": 1000, 
            "page": page
        }
        
        try:
            # Short timeout to fail fast if stuck
            response = requests.get(url, headers=headers, params=params, timeout=10)
            if response.status_code != 200:
                break 

            data = response.json().get('results', [])
            if not data:
                break 
                
            all_results.extend(data)
            
            # Stop if page is not full (end of data)
            if len(data) < 1000:
                break
                
            page += 1
            time.sleep(0.2) # Respect rate limits
            
        except Exception:
            break
            
    return pd.DataFrame(all_results)

print("✅ Helper functions defined.")

✅ Helper functions defined.


In [7]:
# --- CELL 3: MAIN LOOP (UPDATED) ---

# 1. Get the Sensor IDs (Using the ID 3276359 from your successful test)
LOCATION_ID = 3276359
sensor_map = get_location_sensors(LOCATION_ID)

if not sensor_map:
    print("Stopping: No sensors found.")
else:
    print(f"✅ Found {len(sensor_map)} sensors: {list(sensor_map.keys())}")
    
    # 2. Loop by Month
    date_ranges = pd.date_range(start=START_DATE, end=END_DATE, freq='MS')

    for date in date_ranges:
        month_start = date.strftime("%Y-%m-%d")
        next_month = date + pd.DateOffset(months=1)
        
        # Handle current month
        if next_month > datetime.now():
            next_month = datetime.now()
        month_end = next_month.strftime("%Y-%m-%d")
        
        # Don't process future months
        if date > datetime.now():
            continue
            
        filename = f"{OUTPUT_FOLDER}/hcmc_aq_{month_start[:7]}.csv"
        
        # Skip if already downloaded
        if os.path.exists(filename):
            print(f"[SKIP] {month_start[:7]} already exists.")
            continue

        print(f"\nProcessing {month_start[:7]}...", end=" ")
        month_dfs = []

        # 3. Loop through EACH SENSOR (PM2.5, Temp, etc.)
        for param_name, sensor_id in sensor_map.items():
            # Get data for this specific sensor
            df = fetch_sensor_measurements(sensor_id, month_start, month_end)
            
            if not df.empty:
                # Keep only necessary columns
                df['parameter'] = param_name
                
                # Extract date from the nested 'period' object
                # V3 structure is period -> {'datetimeTo': {'local': '...'}}
                df['datetime_local'] = df['period'].apply(
                    lambda x: x.get('datetimeTo', {}).get('local') if isinstance(x, dict) else None
                )
                
                # Fallback if 'local' isn't nested
                if df['datetime_local'].isnull().all():
                     df['datetime_local'] = df['period'].apply(lambda x: x.get('datetimeTo'))

                month_dfs.append(df[['datetime_local', 'value', 'parameter']])
        
        # 4. Combine & Save
        if month_dfs:
            full_month_df = pd.concat(month_dfs, ignore_index=True)
            
            # PIVOT: Turn rows into columns (Time | PM2.5 | Temp | ...)
            full_month_df['datetime_local'] = pd.to_datetime(full_month_df['datetime_local'])
            
            wide_df = full_month_df.pivot_table(
                index='datetime_local', 
                columns='parameter', 
                values='value', 
                aggfunc='mean'
            ).reset_index()
            
            wide_df.to_csv(filename, index=False)
            print(f"-> ✅ Saved {len(wide_df)} rows.")
        else:
            print("-> ⚠️ No data.")

print("\n🎉 Data collection complete!")

✅ Found 6 sensors: ['pm1', 'pm10', 'pm25', 'relativehumidity', 'temperature', 'um003']

Processing 2024-01... -> ⚠️ No data.

Processing 2024-02... -> ⚠️ No data.

Processing 2024-03... -> ⚠️ No data.

Processing 2024-04... -> ⚠️ No data.

Processing 2024-05... -> ⚠️ No data.

Processing 2024-06... -> ⚠️ No data.

Processing 2024-07... -> ⚠️ No data.

Processing 2024-08... -> ⚠️ No data.

Processing 2024-09... -> ⚠️ No data.

Processing 2024-10... -> ⚠️ No data.

Processing 2024-11... -> ✅ Saved 271 rows.

Processing 2024-12... -> ✅ Saved 735 rows.

Processing 2025-01... -> ✅ Saved 255 rows.

Processing 2025-02... -> ✅ Saved 383 rows.

Processing 2025-03... -> ✅ Saved 744 rows.

Processing 2025-04... -> ✅ Saved 720 rows.

Processing 2025-05... -> ✅ Saved 716 rows.

Processing 2025-06... -> ✅ Saved 266 rows.

Processing 2025-07... -> ✅ Saved 744 rows.

Processing 2025-08... -> ✅ Saved 744 rows.

Processing 2025-09... -> ✅ Saved 708 rows.

Processing 2025-10... -> ✅ Saved 738 rows.

Proc

In [8]:
import glob

print("Combining all monthly files...")

# Get all CSV files from the folder
all_files = glob.glob(os.path.join(OUTPUT_FOLDER, "*.csv"))

if all_files:
    # Read all files
    df_list = [pd.read_csv(f, low_memory=False) for f in all_files]
    
    # Concatenate
    final_df = pd.concat(df_list, ignore_index=True)
    
    if 'datetime_local' in final_df.columns:
        # coerce=True will turn invalid dates into NaT (Not a Time) instead of crashing
        final_df['datetime_local'] = pd.to_datetime(final_df['datetime_local'], errors='coerce')
        
        # Sort by date
        final_df = final_df.sort_values(by='datetime_local')
    
    # Save the final 
    final_filename = "Final_HCMC_AirQuality_2024_2026.csv"
    final_df.to_csv(final_filename, index=False)
    
    print(f"SUCCESS! Final dataset saved as: {final_filename}")
    print(f"Total records: {len(final_df)}")
    
    # Print a quick preview of columns
    print(f"Columns: {list(final_df.columns)}")
else:
    print("No CSV files found. Please check if Cell 3 ran correctly.")

Combining all monthly files...
SUCCESS! Final dataset saved as: Final_HCMC_AirQuality_2024_2026.csv
Total records: 9418
Columns: ['datetime_local', 'pm1', 'pm10', 'pm25', 'relativehumidity', 'temperature', 'um003']
